In [ ]:
"""
Analysis script for comparing NeurIPS checklist results across different models.
Creates visualizations and statistics to compare model performance.
"""

import os
import json
import math
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import numpy as np

# ============================================================
# LOCALIZATION
# ============================================================
LANG = "en"  # "en" or "es"

STRINGS = {
    "en": {
        "analysis_title":           "NeurIPS Checklist Analysis",
        "no_dirs":                  "Error: No model directories found!",
        "found_models":             "Found {n} model(s): {names}",
        "loading":                  "Loading results...",
        "n_papers":                 "{model}: {n} papers",
        "creating_df":              "Creating comparison dataframe...",
        "total_comparisons":        "  Total comparisons: {n}",
        "unique_papers":            "  Unique papers: {n}",
        "calculating":              "Calculating statistics...",
        "generating":               "Generating visualizations...",
        "saving":                   "Saving reports...",
        "summary":                  "SUMMARY",
        "models_analyzed":          "Models analyzed: {n}",
        "total_papers":             "Total papers: {n}",
        "best_model":               "Best performing model:",
        "best_model_val":           "  {model}: {acc:.1%} accuracy",
        "easiest_group":            "Easiest question group (highest agreement):",
        "hardest_group":            "Hardest question group (lowest agreement):",
        "easiest_q":                "Easiest question (highest agreement):",
        "hardest_q":                "Hardest question (lowest agreement):",
        "group_val":                "  {name}: {acc:.1%}",
        "results_saved":            "All results saved to: {path}",
        "skip_pairwise":            "⚠  Skipping pairwise agreement: at least 2 models required.",
        # plot labels
        "accuracy_ylabel":          "Accuracy (Agreement Rate)",
        "model_xlabel":             "Model",
        "overall_title":            "Overall Model Accuracy in Checklist Evaluation",
        "heatmap_cbar":             "Agreement Rate",
        "heatmap_title":            "Agreement Rate by Question and Model",
        "heatmap_xlabel":           "Model",
        "heatmap_ylabel":           "Question",
        "difficulty_xlabel":        "Agreement Rate",
        "difficulty_title":         "Questions Ranked by Agreement Rate",
        "confusion_xlabel":         "LLM Answer",
        "confusion_ylabel":         "Author Answer",
        "confusion_suptitle":       "Confusion Matrices — Author Answer vs. LLM Answer",
        "dist_title":               "Distribution of Answers",
        "dist_xlabel":              "Model",
        "dist_ylabel":              "Percentage",
        "dist_legend":              "Answer",
        "authors_label":            "Authors",
        "score_ylabel":             "Number of papers",
        "score_title":              "Per-Paper Score Distribution — Model Comparison",
        "score_cum_ylabel":         "Papers with ≥ this score\n(cumulative)",
        "score_xlabel":             "Score (correct / total questions)",
        "score_legend":             "Model",
        "errbar_ylabel":            "Mean Agreement Rate (± std across models)",
        "errbar_threshold80":       "80% threshold",
        "errbar_threshold50":       "50% threshold",
        "errbar_title":             "Per-Question Accuracy with Cross-Model Variance\n(error bars = std across models)",
        "pairwise_cbar":            "Inter-Model Agreement Rate",
        "pairwise_title":           "Pairwise Model Agreement\n(how often two models give the same answer)",
        "pairwise_xlabel":          "Model",
        "pairwise_ylabel":          "Model",
        "bias_ylabel":              "Δ Proportion vs. Authors",
        "bias_title":               "Answer Bias Relative to Authors\n(positive = model over-uses this answer, negative = under-uses)",
        "bias_legend":              "Model",
        "group_heatmap_title":      "Agreement Rate by Question Group and Model",
        "group_heatmap_xlabel":     "Model",
        "group_heatmap_ylabel":     "Question Group",
        "group_bar_ylabel":         "Agreement Rate",
        "group_bar_title":          "Overall Agreement Rate by Question Group",
        # report
        "report_title":             "NeurIPS CHECKLIST ANALYSIS - STATISTICS REPORT",
        "report_by_model":          "OVERALL ACCURACY BY MODEL",
        "report_by_group":          "ACCURACY BY QUESTION GROUP",
        "report_by_model_group":    "ACCURACY BY MODEL AND QUESTION GROUP",
        "report_best10":            "ACCURACY BY QUESTION (Top 10 Best)",
        "report_worst10":           "ACCURACY BY QUESTION (Top 10 Worst)",
        "report_confusion":         "CONFUSION MATRIX - {model}",
        "report_dataset":           "DATASET INFORMATION",
        "report_papers":            "{model}: {n} papers analyzed",
    },
    "es": {
        "analysis_title":           "Análisis del Checklist de NeurIPS",
        "no_dirs":                  "Error: ¡No se encontraron directorios de modelos!",
        "found_models":             "Se encontraron {n} modelo(s): {names}",
        "loading":                  "Cargando resultados...",
        "n_papers":                 "{model}: {n} artículos",
        "creating_df":              "Creando dataframe de comparación...",
        "total_comparisons":        "  Comparaciones totales: {n}",
        "unique_papers":            "  Artículos únicos: {n}",
        "calculating":              "Calculando estadísticas...",
        "generating":               "Generando visualizaciones...",
        "saving":                   "Guardando informes...",
        "summary":                  "RESUMEN",
        "models_analyzed":          "Modelos analizados: {n}",
        "total_papers":             "Artículos totales: {n}",
        "best_model":               "Modelo con mejor rendimiento:",
        "best_model_val":           "  {model}: {acc:.1%} de acierto",
        "easiest_group":            "Grupo de preguntas más fácil (mayor concordancia):",
        "hardest_group":            "Grupo de preguntas más difícil (menor concordancia):",
        "easiest_q":                "Pregunta más fácil (mayor concordancia):",
        "hardest_q":                "Pregunta más difícil (menor concordancia):",
        "group_val":                "  {name}: {acc:.1%}",
        "results_saved":            "Todos los resultados guardados en: {path}",
        "skip_pairwise":            "⚠  Omitiendo concordancia entre pares: se necesitan al menos 2 modelos.",
        # plot labels
        "accuracy_ylabel":          "Acierto (Tasa de Concordancia)",
        "model_xlabel":             "Modelo",
        "overall_title":            "Acierto General de los Modelos en la Evaluación del Checklist",
        "heatmap_cbar":             "Tasa de Concordancia",
        "heatmap_title":            "Tasa de Concordancia por Pregunta y Modelo",
        "heatmap_xlabel":           "Modelo",
        "heatmap_ylabel":           "Pregunta",
        "difficulty_xlabel":        "Tasa de Concordancia",
        "difficulty_title":         "Preguntas Ordenadas por Tasa de Concordancia",
        "confusion_xlabel":         "Respuesta del LLM",
        "confusion_ylabel":         "Respuesta del Autor",
        "confusion_suptitle":       "Matrices de Confusión — Respuesta del Autor vs. Respuesta del LLM",
        "dist_title":               "Distribución de Respuestas",
        "dist_xlabel":              "Modelo",
        "dist_ylabel":              "Porcentaje",
        "dist_legend":              "Respuesta",
        "authors_label":            "Autores",
        "score_ylabel":             "Número de artículos",
        "score_title":              "Distribución de Puntuaciones por Artículo — Comparación de Modelos",
        "score_cum_ylabel":         "Artículos con ≥ esta puntuación\n(acumulado)",
        "score_xlabel":             "Puntuación (correctas / preguntas totales)",
        "score_legend":             "Modelo",
        "errbar_ylabel":            "Tasa de Concordancia Media (± desv. típica entre modelos)",
        "errbar_threshold80":       "Umbral 80%",
        "errbar_threshold50":       "Umbral 50%",
        "errbar_title":             "Acierto por Pregunta con Varianza entre Modelos\n(barras de error = desviación típica entre modelos)",
        "pairwise_cbar":            "Tasa de Concordancia entre Respuestas",
        "pairwise_title":           "Concordancia entre Pares de Modelos\n(frecuencia con la que dos modelos dan la misma respuesta)",
        "pairwise_xlabel":          "Modelo",
        "pairwise_ylabel":          "Modelo",
        "bias_ylabel":              "Δ Proporción vs. Autores",
        "bias_title":               "Sesgo de Respuesta Respecto a los Autores\n(positivo = el modelo usa esta respuesta más de lo esperado, negativo = menos)",
        "bias_legend":              "Modelo",
        "group_heatmap_title":      "Tasa de Concordancia por Grupo de Preguntas y Modelo",
        "group_heatmap_xlabel":     "Modelo",
        "group_heatmap_ylabel":     "Grupo de Preguntas",
        "group_bar_ylabel":         "Tasa de Concordancia",
        "group_bar_title":          "Tasa de Concordancia General por Grupo de Preguntas",
        # report
        "report_title":             "ANÁLISIS DEL CHECKLIST DE NeurIPS - INFORME DE ESTADÍSTICAS",
        "report_by_model":          "PRECISIÓN GENERAL POR MODELO",
        "report_by_group":          "PRECISIÓN POR GRUPO DE PREGUNTAS",
        "report_by_model_group":    "PRECISIÓN POR MODELO Y GRUPO DE PREGUNTAS",
        "report_best10":            "PRECISIÓN POR PREGUNTA (Top 10 Mejores)",
        "report_worst10":           "PRECISIÓN POR PREGUNTA (Top 10 Peores)",
        "report_confusion":         "MATRIZ DE CONFUSIÓN - {model}",
        "report_dataset":           "INFORMACIÓN DEL CONJUNTO DE DATOS",
        "report_papers":            "{model}: {n} artículos analizados",
    },
}


def t(key, **kwargs):
    """Return the localized string for *key*, formatted with any kwargs."""
    return STRINGS[LANG][key].format(**kwargs)


# ============================================================
# CONFIGURATION
# ============================================================
OUTPUT_BASE_DIR = "neurips_2024_checklist_analysis"
ANALYSIS_OUTPUT_DIR = os.path.join(OUTPUT_BASE_DIR, "analysis_results")

os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# ============================================================
# CHECKLIST QUESTIONS AND GROUPS
# ============================================================
CHECKLIST_QUESTIONS = [
    "Claims",
    "Limitations",
    "Theory Assumptions and Proofs",
    "Experimental Result Reproducibility",
    "Open access to data and code",
    "Experimental Setting/Details",
    "Experiment Statistical Significance",
    "Experiments Compute Resources",
    "Code Of Ethics",
    "Broader Impacts",
    "Safeguards",
    "Licenses for existing assets",
    "New Assets",
    "Crowdsourcing and Research with Human Subjects",
    "Institutional Review Board (IRB) Approvals or Equivalent for Research with Human Subjects",
]

TOTAL_QUESTIONS = len(CHECKLIST_QUESTIONS)

QUESTION_GROUPS = {
    "Authors": [
        "Claims",
        "Limitations",
        "Broader Impacts",
        "Code Of Ethics",
        "Safeguards",
    ],
    "Theory": [
        "Theory Assumptions and Proofs",
    ],
    "Experiments": [
        "Experimental Result Reproducibility",
        "Open access to data and code",
        "Experimental Setting/Details",
        "Experiment Statistical Significance",
        "Experiments Compute Resources",
    ],
    "Assets": [
        "Licenses for existing assets",
        "New Assets",
    ],
    "Human Subjects": [
        "Crowdsourcing and Research with Human Subjects",
        "Institutional Review Board (IRB) Approvals or Equivalent for Research with Human Subjects",
    ],
}

QUESTION_TO_GROUP = {
    question: group
    for group, questions in QUESTION_GROUPS.items()
    for question in questions
}

GROUP_COLORS = {
    "Authors":        "#3498db",
    "Theory":         "#9b59b6",
    "Experiments":    "#2ecc71",
    "Assets":         "#e67e22",
    "Human Subjects": "#e74c3c",
    "Unknown":        "#95a5a6",
}


# ============================================================
# DATA LOADING
# ============================================================

def get_all_model_directories():
    if not os.path.exists(OUTPUT_BASE_DIR):
        return []
    return [
        item for item in os.listdir(OUTPUT_BASE_DIR)
        if os.path.isdir(os.path.join(OUTPUT_BASE_DIR, item))
        and not item.startswith('.')
        and not item.startswith('analysis_results')
    ]


def load_all_results(model_name):
    model_dir = os.path.join(OUTPUT_BASE_DIR, model_name)
    if not os.path.exists(model_dir):
        print(f"Warning: Model directory not found: {model_dir}")
        return []

    results = []
    for filename in os.listdir(model_dir):
        if filename.endswith('.json'):
            json_path = os.path.join(model_dir, filename)
            try:
                with open(json_path, 'r') as f:
                    results.append(json.load(f))
            except Exception as e:
                print(f"Error loading {json_path}: {e}")
    return results


def create_comparison_dataframe(models_data):
    rows = []
    for model_name, results in models_data.items():
        for paper_data in results:
            paper_hash = paper_data.get('paper_hash', 'unknown')
            for question, answers in paper_data.get('questions', {}).items():
                author_ans = answers.get('author_answer', 'Not Found')
                llm_ans   = answers.get('llm_answer',   'Not Found')

                llm_ans    = "NA" if llm_ans    == "N/A" else llm_ans
                author_ans = "NA" if author_ans == "N/A" else author_ans

                if author_ans == 'Not Found' or llm_ans == 'Not Found':
                    print("Not FOUND", paper_hash)
                    continue

                rows.append({
                    'paper_hash':     paper_hash,
                    'model':          model_name,
                    'question':       question,
                    'question_group': QUESTION_TO_GROUP.get(question, 'Unknown'),
                    'author_answer':  author_ans,
                    'llm_answer':     llm_ans,
                    'agreement':      author_ans == llm_ans,
                })
    return pd.DataFrame(rows)


# ============================================================
# STATISTICS
# ============================================================

def calculate_statistics(df):
    stats = {}

    stats['by_model'] = df.groupby('model').agg(
        {'agreement': ['count', 'sum', 'mean']}
    ).round(3)
    stats['by_model'].columns = ['total_comparisons', 'agreements', 'accuracy']

    stats['by_question'] = df.groupby('question').agg(
        {'agreement': ['count', 'sum', 'mean']}
    ).round(3)
    stats['by_question'].columns = ['total_comparisons', 'agreements', 'accuracy']
    stats['by_question'] = stats['by_question'].sort_values('accuracy', ascending=False)

    stats['by_group'] = df.groupby('question_group').agg(
        {'agreement': ['count', 'sum', 'mean']}
    ).round(3)
    stats['by_group'].columns = ['total_comparisons', 'agreements', 'accuracy']
    stats['by_group'] = stats['by_group'].sort_values('accuracy', ascending=False)

    stats['by_model_question'] = (
        df.groupby(['model', 'question'])['agreement'].mean().unstack()
    )
    stats['by_model_group'] = (
        df.groupby(['model', 'question_group'])['agreement'].mean().unstack()
    )

    stats['confusion'] = {}
    for model in df['model'].unique():
        mdf = df[df['model'] == model]
        stats['confusion'][model] = pd.crosstab(
            mdf['author_answer'], mdf['llm_answer'], normalize='index'
        ).round(3)

    return stats


# ============================================================
# PLOTS
# ============================================================

def plot_overall_accuracy(df):
    fig, ax = plt.subplots(figsize=(10, 6))
    accuracy_by_model = df.groupby('model')['agreement'].mean().sort_values(ascending=False)

    bars = ax.bar(range(len(accuracy_by_model)), accuracy_by_model.values)
    ax.set_xticks(range(len(accuracy_by_model)))
    ax.set_xticklabels(accuracy_by_model.index, rotation=45, ha='right')
    ax.set_ylabel(t("accuracy_ylabel"))
    ax.set_xlabel(t("model_xlabel"))
    ax.set_title(t("overall_title"))
    ax.set_ylim([0, 1])

    for bar, val in zip(bars, accuracy_by_model.values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                f'{val:.1%}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'overall_accuracy.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: overall_accuracy.png")


def plot_accuracy_by_question(df):
    fig, ax = plt.subplots(figsize=(14, 10))
    pivot = df.groupby(['question', 'model'])['agreement'].mean().unstack()

    sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn',
                center=0.5, vmin=0, vmax=1, ax=ax,
                cbar_kws={'label': t("heatmap_cbar")})
    ax.set_title(t("heatmap_title"))
    ax.set_xlabel(t("heatmap_xlabel"))
    ax.set_ylabel(t("heatmap_ylabel"))

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'accuracy_by_question_heatmap.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: accuracy_by_question_heatmap.png")


def plot_accuracy_by_group(df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    pivot = df.groupby(['question_group', 'model'])['agreement'].mean().unstack()
    group_order = [g for g in QUESTION_GROUPS if g in pivot.index]
    pivot = pivot.reindex(group_order)

    sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn',
                center=0.5, vmin=0, vmax=1, ax=axes[0],
                cbar_kws={'label': t("heatmap_cbar")})
    axes[0].set_title(t("group_heatmap_title"))
    axes[0].set_xlabel(t("group_heatmap_xlabel"))
    axes[0].set_ylabel(t("group_heatmap_ylabel"))

    group_accuracy = df.groupby('question_group')['agreement'].mean()
    group_accuracy = group_accuracy.reindex([g for g in group_order if g in group_accuracy.index])

    bars = axes[1].bar(
        range(len(group_accuracy)), group_accuracy.values,
        color=[GROUP_COLORS.get(g, '#95a5a6') for g in group_accuracy.index]
    )
    axes[1].set_xticks(range(len(group_accuracy)))
    axes[1].set_xticklabels(group_accuracy.index, rotation=30, ha='right')
    axes[1].set_ylabel(t("group_bar_ylabel"))
    axes[1].set_title(t("group_bar_title"))
    axes[1].set_ylim([0, 1])

    for bar, val in zip(bars, group_accuracy.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                     f'{val:.1%}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'accuracy_by_group.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: accuracy_by_group.png")


def plot_question_difficulty(df):
    fig, ax = plt.subplots(figsize=(12, 8))
    question_accuracy = df.groupby('question')['agreement'].agg(['mean', 'count'])
    question_accuracy = question_accuracy.sort_values('mean')

    colors = [GROUP_COLORS.get(QUESTION_TO_GROUP.get(q, 'Unknown'), '#95a5a6')
              for q in question_accuracy.index]

    bars = ax.barh(range(len(question_accuracy)), question_accuracy['mean'].values, color=colors)
    ax.set_yticks(range(len(question_accuracy)))
    ax.set_yticklabels(question_accuracy.index)
    ax.set_xlabel(t("difficulty_xlabel"))
    ax.set_title(t("difficulty_title"))
    ax.set_xlim([0, 1])

    for bar, val in zip(bars, question_accuracy['mean'].values):
        ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
                f'{val:.1%}', va='center')

    handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=g)
               for g, c in GROUP_COLORS.items() if g != 'Unknown']
    ax.legend(handles=handles, title=t("score_legend"), loc='lower right')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'question_difficulty.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: question_difficulty.png")


def plot_confusion_matrices(df, models):
    n_models = len(models)
    n_cols = min(n_models, 3)
    n_rows = math.ceil(n_models / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6 * n_cols, 5 * n_rows),
                             squeeze=False)

    for idx, model in enumerate(models):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]

        mdf = df[df['model'] == model]
        confusion = pd.crosstab(mdf['author_answer'], mdf['llm_answer'], normalize='index')

        sns.heatmap(confusion, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                    vmin=0, vmax=1, cbar_kws={'label': 'Proportion'})
        ax.set_title(model, fontsize=11, fontweight='bold')
        ax.set_xlabel(t("confusion_xlabel"))
        ax.set_ylabel(t("confusion_ylabel"))

    for idx in range(n_models, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    fig.suptitle(t("confusion_suptitle"), fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'confusion_matrices.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: confusion_matrices.png")


def plot_answer_distribution(df):
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    llm_dist = df.groupby(['model', 'llm_answer']).size().unstack(fill_value=0)
    llm_dist_pct = llm_dist.div(llm_dist.sum(axis=1), axis=0) * 100

    author_dist = df.groupby('author_answer').size()
    author_dist_pct = (author_dist / author_dist.sum() * 100).to_frame().T
    author_dist_pct.index = [t("authors_label")]

    all_cols = llm_dist_pct.columns.union(author_dist_pct.columns)
    combined = pd.concat([
        llm_dist_pct.reindex(columns=all_cols, fill_value=0),
        author_dist_pct.reindex(columns=all_cols, fill_value=0),
    ])

    combined.plot(kind='bar', stacked=True, ax=ax,
                  color=['#95a5a6', '#e74c3c', '#2ecc71'])

    ax.set_title(t("dist_title"))
    ax.set_xlabel(t("dist_xlabel"))
    ax.set_ylabel(t("dist_ylabel"))
    ax.legend(title=t("dist_legend"), bbox_to_anchor=(1.05, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'answer_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: answer_distribution.png")

def plot_accuracy_per_group_per_model(df):
    pivot = df.groupby(['question_group', 'model'])['agreement'].mean().unstack()

    group_order = [g for g in QUESTION_GROUPS if g in pivot.index]
    pivot = pivot.reindex(group_order)

    models    = pivot.columns.tolist()
    n_models  = len(models)
    bar_width = 0.7 / n_models
    x         = np.arange(len(group_order))
    colours   = plt.cm.tab10.colors[:n_models]

    fig, ax = plt.subplots(figsize=(max(10, len(group_order) * 1.8 + 3), 6))

    for i, model in enumerate(models):
        offset = (i - n_models / 2 + 0.5) * bar_width
        bars = ax.bar(
            x + offset, pivot[model].values,
            width=bar_width, label=model,
            color=colours[i], edgecolor='white', linewidth=0.6
        )
        for bar, val in zip(bars, pivot[model].values):
            if not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    val + 0.015,
                    f'{val:.0%}',
                    ha='center', va='bottom', fontsize=7.5
                )

    ax.set_xticks(x)
    ax.set_xticklabels(group_order, rotation=20, ha='right', fontsize=10)
    ax.set_ylabel(t("group_bar_ylabel"))
    ax.set_title(t("group_heatmap_title"))   # reuses existing localized string
    ax.set_ylim(0, 1.12)
    ax.legend(title=t("score_legend"), bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.savefig(
        os.path.join(ANALYSIS_OUTPUT_DIR, 'accuracy_per_group_per_model.png'),
        dpi=300, bbox_inches='tight'
    )
    plt.close()
    print("✓ Saved: accuracy_per_group_per_model.png")
    
def plot_per_paper_score_distribution(df):
    paper_scores = (
        df.groupby(['model', 'paper_hash'])['agreement']
        .agg(correct='sum', total='count')
        .reset_index()
    )
    paper_scores['score_label'] = (
        paper_scores['correct'].astype(int).astype(str) + ' / ' +
        paper_scores['total'].astype(int).astype(str)
    )

    all_totals = sorted(paper_scores['total'].unique(), reverse=True)
    all_labels = [f'{c} / {int(t_)}' for t_ in all_totals for c in range(int(t_), -1, -1)]

    models = sorted(df['model'].unique())
    full_grid = pd.DataFrame(
        [(m, lbl) for m in models for lbl in all_labels],
        columns=['model', 'score_label']
    )
    score_counts = (
        paper_scores.groupby(['model', 'score_label']).size().reset_index(name='num_papers')
    )
    score_counts = full_grid.merge(score_counts, on=['model', 'score_label'], how='left').fillna(0)
    score_counts['num_papers'] = score_counts['num_papers'].astype(int)

    pivot = score_counts.pivot(index='score_label', columns='model', values='num_papers')
    pivot = pivot.reindex(all_labels).loc[lambda d: d.sum(axis=1) > 0]

    n_models  = len(models)
    bar_width = 0.8 / n_models
    x         = np.arange(len(pivot))
    colours   = plt.cm.tab10.colors[:n_models]

    fig, (ax_bars, ax_cum) = plt.subplots(
        2, 1, figsize=(max(14, len(pivot) * 0.7 + 4), 10),
        sharex=True, gridspec_kw={'hspace': 0.08}
    )

    for i, model in enumerate(models):
        offset = (i - n_models / 2 + 0.5) * bar_width
        ax_bars.bar(x + offset, pivot[model].values, width=bar_width,
                    label=model, color=colours[i], edgecolor='white', linewidth=0.6)

    ax_bars.set_ylabel(t("score_ylabel"), fontsize=11)
    ax_bars.set_title(t("score_title"), fontsize=14, pad=14)
    ax_bars.set_ylim(bottom=0)
    ax_bars.tick_params(labelbottom=False)

    for i, model in enumerate(models):
        ax_cum.plot(x, pivot[model].values.cumsum(),
                    color=colours[i], linewidth=2.2, marker='o', markersize=4, label=model)

    ax_cum.set_ylabel(t("score_cum_ylabel"), fontsize=11)
    ax_cum.set_xlabel(t("score_xlabel"), fontsize=11)
    ax_cum.set_xticks(x)
    ax_cum.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=9)
    ax_cum.set_ylim(bottom=0)

    handles, labels = ax_bars.get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(-0.02, 0.5),
               fontsize=9, framealpha=0.95, title=t("score_legend"), title_fontsize=10)
    fig.subplots_adjust(left=0.28)

    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'per_paper_score_distribution.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: per_paper_score_distribution.png")


def plot_per_question_accuracy_with_errorbars(df):
    q_model = df.groupby(['question', 'model'])['agreement'].mean().unstack()
    means   = q_model.mean(axis=1).sort_values(ascending=False)
    stds    = q_model.std(axis=1).reindex(means.index)

    fig, ax = plt.subplots(figsize=(14, 7))
    colors = ['#e74c3c' if m < 0.5 else '#f39c12' if m < 0.8 else '#2ecc71'
              for m in means.values]

    ax.bar(range(len(means)), means.values, yerr=stds.values,
           color=colors, edgecolor='white', linewidth=0.6,
           error_kw=dict(ecolor='#333333', lw=1.5, capsize=4, capthick=1.5))

    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(t("errbar_ylabel"))
    ax.set_ylim(0, 1.12)
    ax.axhline(0.8, color='#27ae60', linestyle='--', linewidth=1, alpha=0.7, label=t("errbar_threshold80"))
    ax.axhline(0.5, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.7, label=t("errbar_threshold50"))
    ax.set_title(t("errbar_title"), fontsize=13)
    ax.legend(fontsize=9)

    for i, (val, std) in enumerate(zip(means.values, stds.values)):
        ax.text(i, val + (std if not np.isnan(std) else 0) + 0.03,
                f'{val:.0%}', ha='center', va='bottom', fontsize=7.5)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'per_question_accuracy_errorbars.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: per_question_accuracy_errorbars.png")


def plot_pairwise_model_agreement(df):
    models = sorted(df['model'].unique())
    n = len(models)

    if n < 2:
        print(t("skip_pairwise"))
        return

    wide = df.pivot_table(
        index=['paper_hash', 'question'],
        columns='model', values='llm_answer', aggfunc='first'
    )

    agreement_matrix = pd.DataFrame(np.nan, index=models, columns=models)
    for i, m1 in enumerate(models):
        for j, m2 in enumerate(models):
            if i == j:
                agreement_matrix.loc[m1, m2] = 1.0
                continue
            if m1 not in wide.columns or m2 not in wide.columns:
                continue
            both = wide[[m1, m2]].dropna()
            if len(both):
                agreement_matrix.loc[m1, m2] = (both[m1] == both[m2]).mean()

    fig, ax = plt.subplots(figsize=(max(6, n * 1.5), max(5, n * 1.4)))
    mask = np.eye(n, dtype=bool)

    sns.heatmap(agreement_matrix.astype(float), annot=True, fmt='.2%',
                cmap='coolwarm', vmin=0, vmax=1, ax=ax, mask=mask,
                cbar_kws={'label': t("pairwise_cbar")}, linewidths=0.5)

    for i in range(n):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='#d5d8dc', lw=0))
        ax.text(i + 0.5, i + 0.5, '—', ha='center', va='center', fontsize=11)

    ax.set_title(t("pairwise_title"), fontsize=13)
    ax.set_xlabel(t("pairwise_xlabel"))
    ax.set_ylabel(t("pairwise_ylabel"))

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'pairwise_model_agreement.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: pairwise_model_agreement.png")


def plot_answer_bias(df):
    answer_cats = sorted(df['author_answer'].unique())
    models      = sorted(df['model'].unique())

    author_props = (df.groupby('author_answer').size() / len(df)).reindex(answer_cats, fill_value=0)

    rows = []
    for model in models:
        mdf      = df[df['model'] == model]
        llm_props = (mdf.groupby('llm_answer').size() / len(mdf)).reindex(answer_cats, fill_value=0)
        for cat in answer_cats:
            rows.append({'model': model, 'answer': cat, 'bias': llm_props[cat] - author_props[cat]})

    pivot     = pd.DataFrame(rows).pivot(index='answer', columns='model', values='bias')
    n_models  = len(models)
    bar_width = 0.7 / n_models
    x         = np.arange(len(answer_cats))
    colours   = plt.cm.tab10.colors[:n_models]

    fig, ax = plt.subplots(figsize=(max(8, n_models * 2.5), 5))
    for i, model in enumerate(models):
        offset = (i - n_models / 2 + 0.5) * bar_width
        ax.bar(x + offset, pivot[model].values, width=bar_width,
               label=model, color=colours[i], edgecolor='white', linewidth=0.5, alpha=0.85)

    ax.axhline(0, color='black', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(answer_cats, fontsize=11)
    ax.set_ylabel(t("bias_ylabel"))
    ax.set_title(t("bias_title"), fontsize=13)
    ax.legend(title=t("bias_legend"), bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.1%}'))

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_OUTPUT_DIR, 'answer_bias.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: answer_bias.png")


# ============================================================
# REPORT
# ============================================================

def save_statistics_report(stats, models_data):
    report_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'statistics_report.txt')

    with open(report_path, 'w') as f:
        f.write("=" * 60 + "\n")
        f.write(t("report_title") + "\n")
        f.write("=" * 60 + "\n\n")

        for section, key in [
            (t("report_by_model"),       'by_model'),
            (t("report_by_group"),       'by_group'),
            (t("report_by_model_group"), 'by_model_group'),
        ]:
            f.write(section + "\n")
            f.write("-" * 60 + "\n")
            f.write(stats[key].to_string())
            f.write("\n\n")

        for section, head in [
            (t("report_best10"), 10),
            (t("report_worst10"), -10),
        ]:
            f.write(section + "\n")
            f.write("-" * 60 + "\n")
            f.write((stats['by_question'].head(head) if head > 0
                     else stats['by_question'].tail(abs(head))).to_string())
            f.write("\n\n")

        for model, confusion in stats['confusion'].items():
            f.write(t("report_confusion", model=model) + "\n")
            f.write("-" * 60 + "\n")
            f.write(confusion.to_string())
            f.write("\n\n")

        f.write(t("report_dataset") + "\n")
        f.write("-" * 60 + "\n")
        for model_name, results in models_data.items():
            f.write(t("report_papers", model=model_name, n=len(results)) + "\n")

    print("✓ Saved: statistics_report.txt")


def export_to_csv(df):
    csv_path = os.path.join(ANALYSIS_OUTPUT_DIR, 'full_comparison.csv')
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved: full_comparison.csv")


# ============================================================
# MAIN
# ============================================================

def analyze_all_models():
    print(f"\n{'=' * 60}")
    print(t("analysis_title"))
    print(f"{'=' * 60}\n")

    models = get_all_model_directories()
    if not models:
        print(t("no_dirs"))
        return

    print(t("found_models", n=len(models), names=', '.join(models)))

    print(t("loading"))
    models_data = {}
    for model in models:
        results = load_all_results(model)
        models_data[model] = results
        print(t("n_papers", model=model, n=len(results)))

    print(t("creating_df"))
    df = create_comparison_dataframe(models_data)
    print(t("total_comparisons", n=f"{len(df):,}"))
    print(t("unique_papers",     n=f"{df['paper_hash'].nunique():,}"))

    print(t("calculating"))
    stats = calculate_statistics(df)

    print(t("generating"))
    plot_overall_accuracy(df)
    plot_accuracy_by_question(df)
    plot_accuracy_by_group(df)
    plot_question_difficulty(df)
    plot_confusion_matrices(df, models)
    plot_answer_distribution(df)
    plot_per_paper_score_distribution(df)
    plot_per_question_accuracy_with_errorbars(df)
    plot_pairwise_model_agreement(df)
    plot_answer_bias(df)
    plot_accuracy_per_group_per_model(df)
    
    print(t("saving"))
    save_statistics_report(stats, models_data)
    export_to_csv(df)

    print("=" * 60)
    print(t("summary"))
    print("=" * 60)
    print(t("models_analyzed", n=len(models)))
    print(t("total_papers",    n=df['paper_hash'].nunique()))
    print(t("total_comparisons", n=f"{len(df):,}"))

    best_model    = stats['by_model']['accuracy'].idxmax()
    best_accuracy = stats['by_model']['accuracy'].max()
    print(t("best_model"))
    print(t("best_model_val", model=best_model, acc=best_accuracy))

    for label_key, idx_fn in [("easiest_group", "idxmax"), ("hardest_group", "idxmin")]:
        name = getattr(stats['by_group']['accuracy'], idx_fn)()
        acc  = getattr(stats['by_group']['accuracy'], idx_fn.replace("idx", ""))()
        print(t(label_key))
        print(t("group_val", name=name, acc=acc))

    for label_key, idx_fn in [("easiest_q", "idxmax"), ("hardest_q", "idxmin")]:
        name = getattr(stats['by_question']['accuracy'], idx_fn)()
        acc  = getattr(stats['by_question']['accuracy'], idx_fn.replace("idx", ""))()
        print(t(label_key))
        print(t("group_val", name=name, acc=acc))

    print(t("results_saved", path=ANALYSIS_OUTPUT_DIR))
    print("=" * 60 + "\n")

    return df, stats


if __name__ == "__main__":
    df, stats = analyze_all_models()